# Models

Looking at the lower level API of Transformers - the models that wrap PyTorch code for the transformers themselves.

This notebook can run on a low-cost or free T4 runtime.


## One more reminder

**Pro-tip:**

In the middle of running a Colab, you might get an error like this:

> Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]

This is a super-misleading error message! Please don't try changing versions of packages...

This actually happens because Google has switched out your Colab runtime, perhaps because Google Colab was too busy. The solution is:

1. Kernel menu >> Disconnect and delete runtime
2. Reload the colab from fresh and Edit menu >> Clear All Outputs
3. Connect to a new T4 using the button at the top right
4. Select "View resources" from the menu on the top right to confirm you have a GPU
5. Rerun the cells in the colab, from the top down, starting with the pip installs

And all should work great - otherwise, ask me!

In [2]:
!pip install -q --upgrade bitsandbytes accelerate

In [3]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

In [4]:
# local VS_Code

import getpass
hf_token = getpass.getpass("Enter HF Token: ")

# Sign in to Hugging Face

1. If you haven't already done so, create a free HuggingFace account at https://huggingface.co and navigate to Settings, then Create a new API token, giving yourself write permissions by clicking on the WRITE tab

2. Press the "key" icon on the side panel to the left, and add a new secret:
`HF_TOKEN = your_token`

3. Execute the cell below to log in.

In [6]:
#hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

### Accessing Llama

Yesterday you should have received approval to use this model:

https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

You can either use that today, or it's faster if you get approval for this model too.

https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

Select this link to see if you need to request approval too. Pick the version of Llama that you want below by commenting out one of these! Or skip Llama altogether.

In [7]:
# instruct models and 1 reasoning model

# Llama 3.1 is larger and you should already be approved
# see here: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

#LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Llama 3.2 is smaller but you might need to request access again
# see here: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

LLAMA = "meta-llama/Llama-3.2-1B-Instruct"

PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [8]:
messages = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"}
  ]

# Accessing Llama 3.1 from Meta

In order to use the fantastic Llama 3.1, Meta does require you to sign their terms of service.

Visit their model instructions page in Hugging Face:
https://huggingface.co/meta-llama/Meta-Llama-3.1-8B

At the top of the page are instructions on how to agree to their terms. If possible, you should use the same email as your huggingface account.

In my experience approval comes in a couple of minutes. Once you've been approved for any 3.1 model, it applies to the whole family of models.

If you have any problems accessing Llama, please see this colab, including some suggestions if you don't get approved by Meta for any reason.

https://colab.research.google.com/drive/1deJO03YZTXUwcq2vzxWbiBhrRuI29Vo8

In [13]:
# Quantization Config - this allows us to load the model into memory and use less memory

# Create a configuration object for quantization (reducing memory usage)
quant_config = BitsAndBytesConfig(
    # Load the model weights in 4-bit precision to drastically reduce memory usage
    load_in_4bit=True,
    
    # Use "double quantization" to quantize the quantization constants themselves (saves ~0.4 bits per parameter)
    bnb_4bit_use_double_quant=True,
    
    # Perform calculations (matrix multiplications) in bfloat16 for better numerical stability
    bnb_4bit_compute_dtype=torch.bfloat16,
    
    # Use NormalFloat 4 (nf4), a special data type optimized for preventing quality loss in neural networks
    bnb_4bit_quant_type="nf4"
)

In [9]:
model_txt = PHI

If the next cell gives you a 403 permissions error, then please check:
1. Are you logged in to HuggingFace? Try running `login()` to check your key works
2. Did you set up your API key with full read and write permissions?
3. If you visit the Llama3.1 page at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B, does it show that you have access to the model near the top?

And work through my Llama troubleshooting colab:

https://colab.research.google.com/drive/1deJO03YZTXUwcq2vzxWbiBhrRuI29Vo8


In [10]:
# Tokenizer

# Load the specific tokenizer for the Llama model (handles text-to-number conversion)
tokenizer = AutoTokenizer.from_pretrained(model_txt)
# Set the padding token to be the same as the End-Of-Sequence (EOS) token
# This is required because Llama usually doesn't have a dedicated pad token by default
tokenizer.pad_token = tokenizer.eos_token
# 1. Convert the list of chat messages (user/assistant roles) into the model's specific string format
# 2. Convert that text into numeric Token IDs (return_tensors="pt" for PyTorch)
# 3. Move the data to the GPU ("cuda") so the model can process it
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [11]:
inputs

tensor([[200021,  60751,    261,  41751,    395,    261,   3435,    328,   4833,
         103481, 200020, 199999]], device='cuda:0')

In [15]:
# The model

model = AutoModelForCausalLM.from_pretrained(model_txt, device_map="auto", quantization_config=quant_config)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# raw output from model
model(inputs)

CausalLMOutputWithPast(loss=None, logits=tensor([[[  14.3516,    8.1953,    8.4766,  ...,    1.9277,    1.9277,
             1.9277],
         [  24.3594,   23.7969,   24.1562,  ...,    7.8008,    7.8008,
             7.8008],
         [  26.4219,   21.7812,   21.2188,  ...,    7.7578,    7.7578,
             7.7578],
         ...,
         [  49.7188,   40.7812,   42.1875,  ...,    9.8203,    9.8203,
             9.8203],
         [-202.5000, -207.2500, -194.8750,  ...,    6.8906,    6.8906,
             6.8906],
         [  19.7188,   18.4219,   25.1719,  ...,   11.9219,   11.9219,
            11.9219]]], device='cuda:0', dtype=torch.float16,
       grad_fn=<UnsafeViewBackward0>), past_key_values=DynamicCache(layers=[DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, D

In [18]:
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")

Memory footprint: 1,588.9 MB


## Looking under the hood at the Transformer model

The next cell prints the HuggingFace `model` object for Llama.

This model object is a Neural Network, implemented with the Python framework PyTorch. The Neural Network uses the architecture invented by Google scientists in 2017: the Transformer architecture.

While we're not going to go deep into the theory, this is an opportunity to get some intuition for what the Transformer actually is.

If you're completely new to Neural Networks, check out my [YouTube intro playlist](https://www.youtube.com/playlist?list=PLWHe-9GP9SMMdl6SLaovUQF2abiLGbMjs) for the foundations.

Now take a look at the layers of the Neural Network that get printed in the next cell. Look out for this:

- It consists of layers
- There's something called "embedding" - this takes tokens and turns them into 4,096 dimensional vectors. We'll learn more about this in Week 5.
- There are then 16 sets of groups of layers (32 for Llama 3.1) called "Decoder layers". Each Decoder layer contains three types of layer: (a) self-attention layers (b) multi-layer perceptron (MLP) layers (c) batch norm layers.
- There is an LM Head layer at the end; this produces the output

Notice the mention that the model has been quantized to 4 bits.

It's not required to go any deeper into the theory at this point, but if you'd like to, I've asked our mutual friend to take this printout and make a tutorial to walk through each layer. This also looks at the dimensions at each point. If you're interested, work through this tutorial after running the next cell:

https://chatgpt.com/canvas/shared/680cbea6de688191a20f350a2293c76b

In [19]:
# Execute this cell and look at what gets printed; investigate the layers

model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1

GEMMA
Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear4bit(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear4bit(in_features=640, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=640, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear4bit(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear4bit(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear4bit(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
        (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
        (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
        (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
      )
    )
    (norm): Gemma3RMSNorm((640,), eps=1e-06)
    (rotary_emb): Gemma3RotaryEmbedding()
    (rotary_emb_local): Gemma3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=640, out_features=262144, bias=False)

### Model Architecture Breakdown

This output confirms that you are running a quantized version of the Llama model (likely **Llama-3.2-1B** given the 16 layers and 2048 hidden dimension).

*   **`LlamaForCausalLM`**: This is the top-level wrapper class for the model, designed for "Causal Language Modeling" (predicting the next token in a sequence).
*   **`model` (`LlamaModel`)**: The core Transformer architecture (excluding the final vocabulary prediction head).
    *   **`embed_tokens`**: The **Embedding Layer**.
        *   `128256`: The vocabulary size (number of unique tokens the model knows).
        *   `2048`: The dimension of the vector representing each token.
    *   **`layers`**: The stack of **16 Transformer Decoder layers** (indices 0-15).
        *   **`self_attn` (`LlamaAttention`)**: The attention mechanism allowing tokens to look at each other.
            *   Notice the `Linear4bit` layers (q, k, v, o projections). This confirms **4-bit quantization** is active, saving huge amounts of memory.
            *   Grouped Query Attention is visible here: `k_proj` and `v_proj` have output features of `512`, while `q_proj` is `2048`. This means there are fewer key/value heads than query heads, which speeds up inference.
        *   **`mlp` (`LlamaMLP`)**: The Feed-Forward Network processing information individually for each token.
            *   It expands dimensions from `2048` to `8192` (`gate_proj`, `up_proj`) and projects back down (`down_proj`).
            *   Uses the `SiLU` activation function.
        *   **`input_layernorm` / `post_attention_layernorm`**: Normalization layers (RMSNorm) that stabilize training and inference.
    *   **`norm`**: The final normalization layer before the output head.
    *   **`rotary_emb`**: Rotary Positional Embeddings (RoPE), which give the model a sense of position/order of tokens.
*   **`lm_head`**: The final **Language Model Head**.
    *   A linear layer that projects the hidden state (`2048`) back to the vocabulary size (`128256`) to calculate probabilities for the next word.

### And if you want to go even deeper into Transformers

In addition to looking at each of the layers in the model, you can actually look at the HuggingFace code that implements Llama using PyTorch.

Here is the HuggingFace Transformers repo:  
https://github.com/huggingface/transformers

And within this, here is the code for Llama 4:  
https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama4/modeling_llama4.py

Obviously it's not neceesary at all to get into this detail - the job of an AI engineer is to select, optimize, fine-tune and apply LLMs rather than to code a transformer in PyTorch. OpenAI, Meta and other frontier labs spent millions building and training these models. But it's a fascinating rabbit hole if you're interested!

In [14]:
# OK, with that, now let's run the model!

outputs = model.generate(inputs, max_new_tokens=80)
outputs[0]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


tensor([128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
            25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
           220,   1627,   3799,    220,   2366,     20,    271, 128009, 128006,
           882, 128007,    271,  41551,    264,  22380,    369,    264,   3130,
           315,   2956,  57116, 128009, 128006,  78191, 128007,    271,   8586,
           596,    832,   1473,  10445,   1550,    279,  12384,    733,    311,
         15419,   1980,  18433,    433,    574,  20558,    311,  30536,   1202,
         21958,    382,     40,   3987,    420,  22380,    364,  19680,   4861,
             6,    701,   3130,    315,    828,  14248,    449,  43214,      0,
        128009], device='cuda:0')

In [15]:
# Well that doesn't make much sense!
# How about this..

tokenizer.decode(outputs[0])

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Dec 2025\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTell a joke for a room of Data Scientists<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nHere's one:\n\nWhy did the algorithm go to therapy?\n\nBecause it was struggling to optimize its emotions.\n\nI hope this joke 'optimizes' your room of data scientists with laughter!<|eot_id|>"

In [1]:
# Clean up memory
# Thank you Kuan L. for helping me get this to properly free up memory!
# If you select "Show Resources" on the top right to see GPU memory, it might not drop down right away
# But it does seem that the memory is available for use by new models in the later code.

del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()

NameError: name 'model' is not defined

## A couple of quick notes on the next block of code:

I'm using a HuggingFace utility called TextStreamer so that results stream back.
To stream results, we simply replace:  
`outputs = model.generate(inputs, max_new_tokens=80)`  
With:  
`streamer = TextStreamer(tokenizer)`  
`outputs = model.generate(inputs, max_new_tokens=80, streamer=streamer)`

Also I've added the argument `add_generation_prompt=True` to my call to create the Chat template. This ensures that Phi generates a response to the question, instead of just predicting how the user prompt continues. Try experimenting with setting this to False to see what happens. You can read about this argument here:

https://huggingface.co/docs/transformers/main/en/chat_templating#what-are-generation-prompts

Thank you to student Piotr B for raising the issue!

In [ ]:
# Wrapping everything in a function - and adding Streaming and generation prompts
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig

def generate(model, messages, quant=True, max_new_tokens=80):
  # Load the tokenizer specific to the model name provided
  tokenizer = AutoTokenizer.from_pretrained(model)
  
  # Ensure the tokenizer has a padding token (using EOS token as fallback)
  tokenizer.pad_token = tokenizer.eos_token
  
  # Convert chat messages to model-specific input format and move to GPU
  # add_generation_prompt=True adds the tokens that signal the assistant to start speaking
  input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
  
  # Create an explicit attention mask (all 1s) to avoid warnings
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")
  
  # Initialize a streamer to print tokens to the console as they are generated
  streamer = TextStreamer(tokenizer)
  
  # Load the model with or without quantization based on the 'quant' flag
  if quant:
    # Load with 4-bit quantization (requires global 'quant_config' variable)
    model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config).to("cuda")
  else:
    # Load standard model (higher memory usage)
    model = AutoModelForCausalLM.from_pretrained(model).to("cuda")
    
  # Generate output. The 'streamer' handles printing text to the console.
  outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)

In [22]:
generate(PHI, messages)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

<|user|>Tell a joke for a room of Data Scientists<|end|><|assistant|>Sure, here's a joke tailored for data scientists:

Why did the data scientist break up with the computer?

Because it kept dumping data on them, and they just couldn't handle the relationship anymore!<|end|>


## Accessing Gemma from Google

A student let me know (thank you, Alex K!) that Google also now requires you to accept their terms in HuggingFace before you use Gemma.

Please visit their model page at this link and confirm you're OK with their terms, so that you're granted access.

https://huggingface.co/google/gemma-3-270m-it

In [11]:
messages = [
    {"role": "user", "content": "Tell a light-hearted joke for a room of physicists"}
  ]
generate(GEMMA, messages, quant=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<bos><start_of_turn>user
Tell a light-hearted joke for a room of physicists<end_of_turn>
<start_of_turn>model
Why don't physicists ever talk to their pets? 

Because they can't keep up!
<end_of_turn>


In [ ]:
generate(QWEN, messages)

In [20]:
generate(DEEPSEEK, messages, quant=False, max_new_tokens=500)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

: 

: 

: 